## Example 1: Basic Environment Interface and TimeStep

Implement a small grid-world `dm_env.Environment` and inspect its specs and the `TimeStep` values that `reset` and `step` return.

In [ ]:
# Example 1: Basic Environment Interface and TimeStep

import dm_env
import numpy as np

# Create a simple custom environment following dm_env interface
class SimpleGridWorld(dm_env.Environment):
    """Simple 3x3 grid world environment"""
    
    def __init__(self):
        self._grid_size = 3
        self._agent_pos = [0, 0]  # Start at top-left
        self._goal_pos = [2, 2]   # Goal at bottom-right
        self._step_count = 0
        self._max_steps = 10
        self._reset_next_step = True
    
    def reset(self):
        """Reset environment to initial state"""
        self._agent_pos = [0, 0]
        self._step_count = 0
        self._reset_next_step = False
        
        observation = self._get_observation()
        return dm_env.restart(observation)
    
    def step(self, action):
        """Take action and return TimeStep"""
        if self._reset_next_step:
            return self.reset()
        
        # Actions: 0=up, 1=right, 2=down, 3=left
        moves = [(-1, 0), (0, 1), (1, 0), (0, -1)]
        if 0 <= action < len(moves):
            dy, dx = moves[action]
            new_y = max(0, min(self._grid_size - 1, self._agent_pos[0] + dy))
            new_x = max(0, min(self._grid_size - 1, self._agent_pos[1] + dx))
            self._agent_pos = [new_y, new_x]
        
        self._step_count += 1
        observation = self._get_observation()
        
        # Check if reached goal
        if self._agent_pos == self._goal_pos:
            self._reset_next_step = True
            return dm_env.termination(reward=10.0, observation=observation)
        
        # Check if max steps reached
        if self._step_count >= self._max_steps:
            self._reset_next_step = True
            return dm_env.termination(reward=-1.0, observation=observation)
        
        # Regular transition
        reward = -0.1  # Small penalty for each step
        return dm_env.transition(reward=reward, observation=observation)
    
    def _get_observation(self):
        """Get current observation"""
        # Return agent position as observation
        return np.array(self._agent_pos, dtype=np.int32)
    
    def observation_spec(self):
        """Specification for observations"""
        return dm_env.specs.Array(
            shape=(2,), dtype=np.int32, name='agent_position'
        )
    
    def action_spec(self):
        """Specification for actions"""
        return dm_env.specs.DiscreteArray(
            num_values=4, name='direction'
        )
    
    def discount_spec(self):
        """Specification for discount factor"""
        return dm_env.specs.BoundedArray(
            shape=(), dtype=np.float32, minimum=0.0, maximum=1.0, name='discount'
        )
    
    def reward_spec(self):
        """Specification for rewards"""
        return dm_env.specs.Array(
            shape=(), dtype=np.float32, name='reward'
        )

print("=== Basic Environment Interface Demo ===")

# Create and test the environment
env = SimpleGridWorld()

print("Environment specifications:")
print(f"  Observation spec: {env.observation_spec()}")
print(f"  Action spec: {env.action_spec()}")
print(f"  Reward spec: {env.reward_spec()}")
print(f"  Discount spec: {env.discount_spec()}")

# Reset environment and examine initial TimeStep
timestep = env.reset()
print(f"\nInitial timestep:")
print(f"  Step type: {timestep.step_type}")
print(f"  Reward: {timestep.reward}")
print(f"  Discount: {timestep.discount}")
print(f"  Observation: {timestep.observation}")

# Take some actions and observe transitions
actions = [1, 2, 1, 2]  # right, down, right, down -> should reach goal
print(f"\nTaking actions: {actions}")

for i, action in enumerate(actions):
    timestep = env.step(action)
    print(f"  Step {i+1}: action={action}")
    print(f"    Step type: {timestep.step_type}")
    print(f"    Reward: {timestep.reward}")
    print(f"    Observation: {timestep.observation}")
    print(f"    Is last: {timestep.last()}")
    
    if timestep.last():
        print(f"    Episode terminated!")
        break

print("✓ Basic environment interface and TimeStep demonstrated!")

## Example 2: Observation and Action Specifications

Demonstrate comprehensive specification system for observations and actions.

In [ ]:
# Example 2: Observation and action specifications

import dm_env
import numpy as np

# Demonstrate various specification types
class RobotControlEnvironment(dm_env.Environment):
    """Robot control environment with complex observations and actions"""
    
    def __init__(self):
        self._robot_state = {
            'joint_angles': np.array([0.0, 0.0, 0.0], dtype=np.float32),
            'joint_velocities': np.array([0.0, 0.0, 0.0], dtype=np.float32),
            'end_effector_pos': np.array([0.5, 0.0, 0.2], dtype=np.float32),
            'gripper_state': 0,  # 0=open, 1=closed
            'battery_level': 100.0,
            'camera_image': np.zeros((64, 64, 3), dtype=np.uint8)
        }
        self._step_count = 0
        self._reset_next_step = True
    
    def reset(self):
        """Reset robot to initial state"""
        self._robot_state['joint_angles'] = np.array([0.0, 0.0, 0.0], dtype=np.float32)
        self._robot_state['joint_velocities'] = np.array([0.0, 0.0, 0.0], dtype=np.float32)
        self._robot_state['end_effector_pos'] = np.array([0.5, 0.0, 0.2], dtype=np.float32)
        self._robot_state['gripper_state'] = 0
        self._robot_state['battery_level'] = 100.0
        self._robot_state['camera_image'] = np.random.randint(0, 256, (64, 64, 3), dtype=np.uint8)
        self._step_count = 0
        self._reset_next_step = False
        
        return dm_env.restart(self._get_observation())
    
    def step(self, action):
        """Execute robot action"""
        if self._reset_next_step:
            return self.reset()
        
        # Action contains: joint_torques, gripper_command, target_velocity
        joint_torques = action['joint_torques']
        gripper_cmd = action['gripper_command']
        target_vel = action['target_velocity']
        
        # Update robot state (simplified physics)
        self._robot_state['joint_velocities'] += joint_torques * 0.1
        self._robot_state['joint_angles'] += self._robot_state['joint_velocities'] * 0.1
        self._robot_state['gripper_state'] = int(gripper_cmd > 0.5)
        self._robot_state['battery_level'] = max(0, self._robot_state['battery_level'] - 0.5)
        
        # Update end effector position (simple forward kinematics approximation)
        angles = self._robot_state['joint_angles']
        self._robot_state['end_effector_pos'] = np.array([
            0.5 + np.sum(np.cos(angles)) * 0.3,
            np.sum(np.sin(angles)) * 0.3,
            0.2 + angles[2] * 0.1
        ], dtype=np.float32)
        
        # Generate new camera image
        self._robot_state['camera_image'] = np.random.randint(0, 256, (64, 64, 3), dtype=np.uint8)
        
        self._step_count += 1
        observation = self._get_observation()
        
        # Simple reward: negative if battery too low
        reward = -10.0 if self._robot_state['battery_level'] < 10 else 0.1
        
        if self._step_count >= 100 or self._robot_state['battery_level'] <= 0:
            self._reset_next_step = True
            return dm_env.termination(reward=reward, observation=observation)
        
        return dm_env.transition(reward=reward, observation=observation)
    
    def _get_observation(self):
        """Get complex multi-modal observation"""
        return {
            'proprioception': {
                'joint_angles': self._robot_state['joint_angles'].copy(),
                'joint_velocities': self._robot_state['joint_velocities'].copy(),
                'end_effector_pos': self._robot_state['end_effector_pos'].copy()
            },
            'gripper_state': np.array([self._robot_state['gripper_state']], dtype=np.int32),
            'battery_level': np.array([self._robot_state['battery_level']], dtype=np.float32),
            'camera_image': self._robot_state['camera_image'].copy(),
            'task_progress': np.array([self._step_count / 100.0], dtype=np.float32)
        }
    
    def observation_spec(self):
        """Complex nested observation specification"""
        return {
            'proprioception': {
                'joint_angles': dm_env.specs.BoundedArray(
                    shape=(3,), dtype=np.float32, 
                    minimum=-np.pi, maximum=np.pi, name='joint_angles'
                ),
                'joint_velocities': dm_env.specs.BoundedArray(
                    shape=(3,), dtype=np.float32,
                    minimum=-10.0, maximum=10.0, name='joint_velocities'
                ),
                'end_effector_pos': dm_env.specs.BoundedArray(
                    shape=(3,), dtype=np.float32,
                    minimum=[-1.0, -1.0, 0.0], maximum=[1.0, 1.0, 0.5],
                    name='end_effector_position'
                )
            },
            'gripper_state': dm_env.specs.DiscreteArray(
                num_values=2, name='gripper_open_closed'
            ),
            'battery_level': dm_env.specs.BoundedArray(
                shape=(1,), dtype=np.float32,
                minimum=0.0, maximum=100.0, name='battery_percentage'
            ),
            'camera_image': dm_env.specs.Array(
                shape=(64, 64, 3), dtype=np.uint8, name='rgb_camera'
            ),
            'task_progress': dm_env.specs.BoundedArray(
                shape=(1,), dtype=np.float32,
                minimum=0.0, maximum=1.0, name='completion_ratio'
            )
        }
    
    def action_spec(self):
        """Complex nested action specification"""
        return {
            'joint_torques': dm_env.specs.BoundedArray(
                shape=(3,), dtype=np.float32,
                minimum=-5.0, maximum=5.0, name='motor_torques'
            ),
            'gripper_command': dm_env.specs.BoundedArray(
                shape=(), dtype=np.float32,
                minimum=0.0, maximum=1.0, name='gripper_control'
            ),
            'target_velocity': dm_env.specs.BoundedArray(
                shape=(3,), dtype=np.float32,
                minimum=-2.0, maximum=2.0, name='velocity_setpoint'
            )
        }

print("=== Complex Specification System Demo ===")

robot_env = RobotControlEnvironment()

# Display observation specifications
print("Observation Specifications:")
obs_spec = robot_env.observation_spec()
for key, spec in obs_spec.items():
    if isinstance(spec, dict):
        print(f"  {key} (nested):")
        for sub_key, sub_spec in spec.items():
            print(f"    {sub_key}: shape={sub_spec.shape}, dtype={sub_spec.dtype}")
            if hasattr(sub_spec, 'minimum') and hasattr(sub_spec, 'maximum'):
                print(f"      bounds=[{sub_spec.minimum}, {sub_spec.maximum}]")
    else:
        print(f"  {key}: shape={spec.shape}, dtype={spec.dtype}")
        if hasattr(spec, 'minimum') and hasattr(spec, 'maximum'):
            print(f"    bounds=[{spec.minimum}, {spec.maximum}]")

# Display action specifications
print(f"\nAction Specifications:")
action_spec = robot_env.action_spec()
for key, spec in action_spec.items():
    print(f"  {key}: shape={spec.shape}, dtype={spec.dtype}")
    print(f"    bounds=[{spec.minimum}, {spec.maximum}]")

# Test with valid actions
print(f"\nTesting with valid actions:")
timestep = robot_env.reset()

# Create valid action according to specification
valid_action = {
    'joint_torques': np.array([1.0, -0.5, 0.2], dtype=np.float32),
    'gripper_command': np.array(0.8, dtype=np.float32),
    'target_velocity': np.array([0.1, 0.0, -0.1], dtype=np.float32)
}

print(f"Taking action: {valid_action}")
timestep = robot_env.step(valid_action)

print(f"Observation keys: {list(timestep.observation.keys())}")
print(f"Proprioception joint angles: {timestep.observation['proprioception']['joint_angles']}")
print(f"Battery level: {timestep.observation['battery_level']}")
print(f"Camera image shape: {timestep.observation['camera_image'].shape}")

# Demonstrate spec validation
print(f"\n=== Specification Validation Demo ===")

def validate_observation(obs, spec):
    """Validate observation against specification"""
    errors = []
    
    for key, expected_spec in spec.items():
        if key not in obs:
            errors.append(f"Missing key: {key}")
            continue
            
        if isinstance(expected_spec, dict):
            # Nested specification
            nested_errors = validate_observation(obs[key], expected_spec)
            errors.extend([f"{key}.{err}" for err in nested_errors])
        else:
            # Single specification
            value = obs[key]
            if value.shape != expected_spec.shape:
                errors.append(f"{key}: shape mismatch {value.shape} != {expected_spec.shape}")
            if value.dtype != expected_spec.dtype:
                errors.append(f"{key}: dtype mismatch {value.dtype} != {expected_spec.dtype}")
    
    return errors

# Validate current observation
observation = timestep.observation
spec_errors = validate_observation(observation, obs_spec)

if spec_errors:
    print("Specification violations:")
    for error in spec_errors:
        print(f"  ❌ {error}")
else:
    print("✅ All observations conform to specifications")

print("✓ Complex observation and action specifications demonstrated!")

## Example 3: TimeStep Types and Environment Lifecycle

Demonstrate different TimeStep types and complete environment lifecycle management.

In [ ]:
# Example 3: TimeStep types and environment lifecycle

import dm_env
import numpy as np

# Game environment demonstrating all TimeStep types
class NumberGuessingGame(dm_env.Environment):
    """Number guessing game showing all TimeStep types"""
    
    def __init__(self, max_range=10):
        self._max_range = max_range
        self._target_number = None
        self._guesses_left = 3
        self._total_guesses = 0
        self._game_over = True
        self._game_won = False
        
    def reset(self):
        """Start new game"""
        self._target_number = np.random.randint(1, self._max_range + 1)
        self._guesses_left = 3
        self._total_guesses = 0
        self._game_over = False
        self._game_won = False
        
        observation = self._get_observation()
        # Return RESTART timestep
        return dm_env.restart(observation)
    
    def step(self, action):
        """Make a guess"""
        if self._game_over:
            return self.reset()
        
        guess = int(action)
        self._total_guesses += 1
        self._guesses_left -= 1
        
        observation = self._get_observation()
        
        # Check if guess is correct
        if guess == self._target_number:
            self._game_over = True
            self._game_won = True
            reward = 10.0 + self._guesses_left  # Bonus for fewer guesses
            # Return TERMINATION timestep (success)
            return dm_env.termination(reward=reward, observation=observation)
        
        # Check if out of guesses
        if self._guesses_left <= 0:
            self._game_over = True
            reward = -5.0  # Penalty for losing
            # Return TERMINATION timestep (failure)
            return dm_env.termination(reward=reward, observation=observation)
        
        # Game continues - provide hint reward
        if guess < self._target_number:
            hint_reward = -0.5  # Small penalty, but hint that guess is too low
        else:
            hint_reward = -0.3  # Small penalty, but hint that guess is too high
        
        # Return MID timestep (game continues)
        return dm_env.transition(reward=hint_reward, observation=observation)
    
    def _get_observation(self):
        """Get current game state"""
        return {
            'guesses_left': np.array([self._guesses_left], dtype=np.int32),
            'total_guesses': np.array([self._total_guesses], dtype=np.int32),
            'game_over': np.array([self._game_over], dtype=bool),
            'last_guess_result': np.array([0], dtype=np.int32)  # 0=continue, 1=too_low, 2=too_high, 3=correct
        }
    
    def observation_spec(self):
        return {
            'guesses_left': dm_env.specs.BoundedArray(
                shape=(1,), dtype=np.int32, minimum=0, maximum=3, name='remaining_guesses'
            ),
            'total_guesses': dm_env.specs.BoundedArray(
                shape=(1,), dtype=np.int32, minimum=0, maximum=10, name='total_attempts'
            ),
            'game_over': dm_env.specs.Array(
                shape=(1,), dtype=bool, name='is_finished'
            ),
            'last_guess_result': dm_env.specs.DiscreteArray(
                num_values=4, name='hint_state'
            )
        }
    
    def action_spec(self):
        return dm_env.specs.BoundedArray(
            shape=(), dtype=np.int32, minimum=1, maximum=self._max_range, name='number_guess'
        )

print("=== TimeStep Types and Lifecycle Demo ===")

game = NumberGuessingGame(max_range=5)

def analyze_timestep(timestep, step_num=None):
    """Analyze and display timestep information"""
    step_info = f"Step {step_num}: " if step_num is not None else ""
    print(f"{step_info}TimeStep Analysis:")
    print(f"  Step type: {timestep.step_type}")
    print(f"  Reward: {timestep.reward}")
    print(f"  Discount: {timestep.discount}")
    print(f"  Is first: {timestep.first()}")
    print(f"  Is mid: {timestep.mid()}")
    print(f"  Is last: {timestep.last()}")
    
    # Observation details
    obs = timestep.observation
    print(f"  Observation:")
    print(f"    Guesses left: {obs['guesses_left'][0]}")
    print(f"    Total guesses: {obs['total_guesses'][0]}")
    print(f"    Game over: {obs['game_over'][0]}")

# Play complete game showing all timestep types
print("Starting new game...")
timestep = game.reset()
analyze_timestep(timestep, "Reset")

# Make some guesses
guesses = [2, 4, 1, 3, 5]  # Will likely hit different scenarios

for i, guess in enumerate(guesses):
    if timestep.last():
        print(f"\nGame ended, starting new game...")
        timestep = game.reset()
        analyze_timestep(timestep, "New Reset")
    
    print(f"\nMaking guess: {guess}")
    timestep = game.step(guess)
    analyze_timestep(timestep, i+1)
    
    if timestep.last():
        result = "Won!" if timestep.reward > 0 else "Lost!"
        print(f"  Game result: {result}")
        break

# Demonstrate TimeStep creation utilities
print(f"\n=== TimeStep Creation Utilities ===")

# Example observations and rewards
sample_obs = np.array([1, 2, 3])
sample_reward = 5.0

# Create different timestep types
restart_step = dm_env.restart(sample_obs)
print(f"Restart TimeStep:")
print(f"  Type: {restart_step.step_type} (value: {restart_step.step_type.value})")
print(f"  Reward: {restart_step.reward}")
print(f"  Discount: {restart_step.discount}")

transition_step = dm_env.transition(reward=sample_reward, observation=sample_obs, discount=0.99)
print(f"\nTransition TimeStep:")
print(f"  Type: {transition_step.step_type} (value: {transition_step.step_type.value})")
print(f"  Reward: {transition_step.reward}")
print(f"  Discount: {transition_step.discount}")

termination_step = dm_env.termination(reward=sample_reward, observation=sample_obs)
print(f"\nTermination TimeStep:")
print(f"  Type: {termination_step.step_type} (value: {termination_step.step_type.value})")
print(f"  Reward: {termination_step.reward}")
print(f"  Discount: {termination_step.discount}")

# Demonstrate timestep checking methods
print(f"\n=== TimeStep Checking Methods ===")
timesteps = [
    ("Restart", restart_step),
    ("Transition", transition_step), 
    ("Termination", termination_step)
]

for name, ts in timesteps:
    print(f"{name} TimeStep:")
    print(f"  first(): {ts.first()}")
    print(f"  mid(): {ts.mid()}")
    print(f"  last(): {ts.last()}")

print("✓ TimeStep types and environment lifecycle demonstrated!")

## Example 4: Environment Wrappers and Extensions

Demonstrate environment wrappers for modifying behavior and adding functionality.

In [ ]:
# Example 4: Environment wrappers and extensions

import dm_env
import numpy as np
from typing import Any

# Base wrapper class
class EnvironmentWrapper(dm_env.Environment):
    """Base wrapper that delegates to wrapped environment"""
    
    def __init__(self, environment):
        self._environment = environment
    
    def reset(self):
        return self._environment.reset()
    
    def step(self, action):
        return self._environment.step(action)
    
    def observation_spec(self):
        return self._environment.observation_spec()
    
    def action_spec(self):
        return self._environment.action_spec()
    
    def reward_spec(self):
        return self._environment.reward_spec()
    
    def discount_spec(self):
        return self._environment.discount_spec()

# Reward scaling wrapper
class RewardScalingWrapper(EnvironmentWrapper):
    """Wrapper that scales rewards by a constant factor"""
    
    def __init__(self, environment, scale_factor=1.0):
        super().__init__(environment)
        self._scale_factor = scale_factor
    
    def step(self, action):
        timestep = self._environment.step(action)
        
        if timestep.reward is not None:
            scaled_reward = timestep.reward * self._scale_factor
        else:
            scaled_reward = timestep.reward
        
        # Create new timestep with scaled reward
        if timestep.first():
            return dm_env.restart(timestep.observation)
        elif timestep.last():
            return dm_env.termination(
                reward=scaled_reward, 
                observation=timestep.observation
            )
        else:
            return dm_env.transition(
                reward=scaled_reward,
                observation=timestep.observation,
                discount=timestep.discount
            )

# Action repeat wrapper
class ActionRepeatWrapper(EnvironmentWrapper):
    """Wrapper that repeats actions multiple times"""
    
    def __init__(self, environment, repeat_count=2):
        super().__init__(environment)
        self._repeat_count = repeat_count
    
    def step(self, action):
        total_reward = 0.0
        
        for _ in range(self._repeat_count):
            timestep = self._environment.step(action)
            
            if timestep.reward is not None:
                total_reward += timestep.reward
            
            # If episode ends, return early
            if timestep.last():
                if timestep.reward is not None:
                    total_reward = timestep.reward  # Use final reward only
                return dm_env.termination(
                    reward=total_reward,
                    observation=timestep.observation
                )
        
        # Return final timestep with accumulated reward
        return dm_env.transition(
            reward=total_reward,
            observation=timestep.observation,
            discount=timestep.discount
        )

# Observation normalization wrapper
class ObservationNormalizationWrapper(EnvironmentWrapper):
    """Wrapper that normalizes observations"""
    
    def __init__(self, environment, obs_key=None):
        super().__init__(environment)
        self._obs_key = obs_key
        self._stats = {'mean': 0.0, 'std': 1.0, 'count': 0}
    
    def _normalize_observation(self, observation):
        """Normalize observation using running statistics"""
        if self._obs_key and isinstance(observation, dict):
            # Handle dict observations
            if self._obs_key in observation:
                raw_obs = observation[self._obs_key]
                normalized_obs = observation.copy()
                normalized_obs[self._obs_key] = self._normalize_array(raw_obs)
                return normalized_obs
            else:
                return observation
        elif isinstance(observation, np.ndarray):
            # Handle array observations
            return self._normalize_array(observation)
        else:
            # Return as-is for other types
            return observation
    
    def _normalize_array(self, array):
        """Normalize numpy array"""
        # Update running statistics (simplified)
        self._stats['count'] += 1
        if self._stats['count'] == 1:
            self._stats['mean'] = np.mean(array)
            self._stats['std'] = max(np.std(array), 1e-8)
        else:
            # Simple exponential moving average
            alpha = 0.01
            new_mean = np.mean(array)
            new_std = np.std(array)
            
            self._stats['mean'] = (1 - alpha) * self._stats['mean'] + alpha * new_mean
            self._stats['std'] = (1 - alpha) * self._stats['std'] + alpha * new_std
            self._stats['std'] = max(self._stats['std'], 1e-8)
        
        # Normalize
        normalized = (array - self._stats['mean']) / self._stats['std']
        return normalized.astype(array.dtype)
    
    def reset(self):
        timestep = self._environment.reset()
        normalized_obs = self._normalize_observation(timestep.observation)
        return dm_env.restart(normalized_obs)
    
    def step(self, action):
        timestep = self._environment.step(action)
        normalized_obs = self._normalize_observation(timestep.observation)
        
        if timestep.first():
            return dm_env.restart(normalized_obs)
        elif timestep.last():
            return dm_env.termination(
                reward=timestep.reward,
                observation=normalized_obs
            )
        else:
            return dm_env.transition(
                reward=timestep.reward,
                observation=normalized_obs,
                discount=timestep.discount
            )

# Episode statistics wrapper
class EpisodeStatsWrapper(EnvironmentWrapper):
    """Wrapper that tracks episode statistics"""
    
    def __init__(self, environment):
        super().__init__(environment)
        self.reset_stats()
    
    def reset_stats(self):
        self._episode_return = 0.0
        self._episode_length = 0
        self._episodes_completed = 0
        self._all_returns = []
    
    def reset(self):
        # If we're resetting after an episode, record stats
        if self._episode_length > 0:
            self._all_returns.append(self._episode_return)
            self._episodes_completed += 1
        
        # Reset for new episode
        self._episode_return = 0.0
        self._episode_length = 0
        
        return self._environment.reset()
    
    def step(self, action):
        timestep = self._environment.step(action)
        
        # Update stats
        self._episode_length += 1
        if timestep.reward is not None:
            self._episode_return += timestep.reward
        
        # If episode ended, record final stats
        if timestep.last():
            self._all_returns.append(self._episode_return)
            self._episodes_completed += 1
        
        return timestep
    
    def get_statistics(self):
        """Get episode statistics"""
        if not self._all_returns:
            return {
                'episodes_completed': 0,
                'mean_return': 0.0,
                'std_return': 0.0,
                'min_return': 0.0,
                'max_return': 0.0,
                'current_episode_return': self._episode_return,
                'current_episode_length': self._episode_length
            }
        
        returns = np.array(self._all_returns)
        return {
            'episodes_completed': self._episodes_completed,
            'mean_return': np.mean(returns),
            'std_return': np.std(returns),
            'min_return': np.min(returns),
            'max_return': np.max(returns),
            'current_episode_return': self._episode_return,
            'current_episode_length': self._episode_length
        }

print("=== Environment Wrappers Demo ===")

# Create base environment (reuse SimpleGridWorld from Example 1)
class SimpleGridWorld(dm_env.Environment):
    def __init__(self):
        self._grid_size = 3
        self._agent_pos = [0, 0]
        self._goal_pos = [2, 2]
        self._step_count = 0
        self._max_steps = 10
        self._reset_next_step = True
    
    def reset(self):
        self._agent_pos = [0, 0]
        self._step_count = 0
        self._reset_next_step = False
        observation = np.array(self._agent_pos, dtype=np.float32)
        return dm_env.restart(observation)
    
    def step(self, action):
        if self._reset_next_step:
            return self.reset()
        
        moves = [(-1, 0), (0, 1), (1, 0), (0, -1)]
        if 0 <= action < len(moves):
            dy, dx = moves[action]
            new_y = max(0, min(self._grid_size - 1, self._agent_pos[0] + dy))
            new_x = max(0, min(self._grid_size - 1, self._agent_pos[1] + dx))
            self._agent_pos = [new_y, new_x]
        
        self._step_count += 1
        observation = np.array(self._agent_pos, dtype=np.float32)
        
        if self._agent_pos == self._goal_pos:
            self._reset_next_step = True
            return dm_env.termination(reward=10.0, observation=observation)
        
        if self._step_count >= self._max_steps:
            self._reset_next_step = True
            return dm_env.termination(reward=-1.0, observation=observation)
        
        return dm_env.transition(reward=-0.1, observation=observation)
    
    def observation_spec(self):
        return dm_env.specs.Array(shape=(2,), dtype=np.float32, name='position')
    
    def action_spec(self):
        return dm_env.specs.DiscreteArray(num_values=4, name='direction')

# Test different wrappers
base_env = SimpleGridWorld()

# Create wrapped environments
reward_scaled_env = RewardScalingWrapper(base_env, scale_factor=2.0)
action_repeat_env = ActionRepeatWrapper(base_env, repeat_count=2)
normalized_env = ObservationNormalizationWrapper(base_env)
stats_env = EpisodeStatsWrapper(base_env)

print("Testing reward scaling wrapper:")
timestep = reward_scaled_env.reset()
timestep = reward_scaled_env.step(1)  # Move right
print(f"  Original reward would be -0.1, scaled reward: {timestep.reward}")

print(f"\nTesting action repeat wrapper:")
timestep = action_repeat_env.reset()
initial_pos = timestep.observation.copy()
timestep = action_repeat_env.step(1)  # Move right (repeated)
final_pos = timestep.observation
print(f"  Initial position: {initial_pos}")
print(f"  Final position after repeated action: {final_pos}")
print(f"  Accumulated reward: {timestep.reward}")

print(f"\nTesting normalization wrapper:")
timestep = normalized_env.reset()
print(f"  Original observation: {base_env.reset().observation}")
print(f"  Normalized observation: {timestep.observation}")

print(f"\nTesting statistics wrapper:")
# Run a few episodes to collect stats
for episode in range(3):
    timestep = stats_env.reset()
    step_count = 0
    while not timestep.last() and step_count < 15:
        action = np.random.randint(0, 4)  # Random action
        timestep = stats_env.step(action)
        step_count += 1

stats = stats_env.get_statistics()
print(f"  Episode statistics:")
print(f"    Episodes completed: {stats['episodes_completed']}")
print(f"    Mean return: {stats['mean_return']:.3f}")
print(f"    Std return: {stats['std_return']:.3f}")
print(f"    Min return: {stats['min_return']:.3f}")
print(f"    Max return: {stats['max_return']:.3f}")

print("✓ Environment wrappers and extensions demonstrated!")

## Example 5: Production Agent-Environment Loop

Demonstrate a complete production-ready agent-environment interaction loop with logging and metrics.

In [ ]:
# Example 5: Production agent-environment loop

import dm_env
import numpy as np
import time
from typing import Dict, List, Any
import logging

# Production-ready environment with comprehensive monitoring
class ProductionEnvironment(dm_env.Environment):
    """Production environment with monitoring and validation"""
    
    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.name = config.get('name', 'ProductionEnv')
        self.max_episode_steps = config.get('max_episode_steps', 100)
        self.target_reward = config.get('target_reward', 100.0)
        
        # State tracking
        self._state = np.array([0.0, 0.0], dtype=np.float32)
        self._target = np.array([1.0, 1.0], dtype=np.float32)
        self._step_count = 0
        self._episode_count = 0
        self._reset_next_step = True
        
        # Monitoring
        self._episode_rewards = []
        self._episode_lengths = []
        self._action_counts = {i: 0 for i in range(4)}
        
        # Logging setup
        self.logger = self._setup_logger()
        
    def _setup_logger(self):
        """Setup logging for environment"""
        logger = logging.getLogger(f'{self.name}')
        logger.setLevel(logging.INFO)
        
        if not logger.handlers:
            handler = logging.StreamHandler()
            formatter = logging.Formatter(
                '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
            )
            handler.setFormatter(formatter)
            logger.addHandler(handler)
        
        return logger
    
    def reset(self):
        """Reset environment with logging and validation"""
        if self._step_count > 0:
            # Log episode completion
            episode_reward = sum(self._episode_rewards[-self._step_count:]) if self._episode_rewards else 0
            self._episode_lengths.append(self._step_count)
            
            self.logger.info(
                f"Episode {self._episode_count} completed: "
                f"reward={episode_reward:.3f}, length={self._step_count}"
            )
        
        # Reset state
        self._state = np.random.uniform(-0.5, 0.5, size=2).astype(np.float32)
        self._target = np.random.uniform(0.5, 1.5, size=2).astype(np.float32)
        self._step_count = 0
        self._episode_count += 1
        self._reset_next_step = False
        
        observation = self._get_observation()
        
        # Validate observation
        self._validate_observation(observation)
        
        self.logger.debug(f"Environment reset: state={self._state}, target={self._target}")
        return dm_env.restart(observation)
    
    def step(self, action):
        """Step environment with validation and monitoring"""
        if self._reset_next_step:
            return self.reset()
        
        # Validate action
        self._validate_action(action)
        
        # Record action
        self._action_counts[int(action)] += 1
        
        # Apply action
        action_effects = {
            0: np.array([0.0, 0.1]),   # Up
            1: np.array([0.1, 0.0]),   # Right  
            2: np.array([0.0, -0.1]),  # Down
            3: np.array([-0.1, 0.0])   # Left
        }
        
        if action in action_effects:
            self._state += action_effects[action]
        
        # Add noise
        self._state += np.random.normal(0, 0.01, size=2)
        
        # Clip to bounds
        self._state = np.clip(self._state, -2.0, 2.0)
        
        self._step_count += 1
        observation = self._get_observation()
        
        # Calculate reward
        distance_to_target = np.linalg.norm(self._state - self._target)
        reward = max(0, 1.0 - distance_to_target)  # Reward decreases with distance
        
        # Check termination conditions
        if distance_to_target < 0.1:
            # Success
            self._reset_next_step = True
            reward += 10.0  # Bonus for reaching target
            self._episode_rewards.append(reward)
            
            self.logger.info(f"Target reached! Final reward: {reward:.3f}")
            return dm_env.termination(reward=reward, observation=observation)
        
        elif self._step_count >= self.max_episode_steps:
            # Timeout
            self._reset_next_step = True
            reward -= 5.0  # Penalty for timeout
            self._episode_rewards.append(reward)
            
            self.logger.info(f"Episode timeout. Final reward: {reward:.3f}")
            return dm_env.termination(reward=reward, observation=observation)
        
        else:
            # Continue episode
            self._episode_rewards.append(reward)
            return dm_env.transition(reward=reward, observation=observation, discount=0.99)
    
    def _get_observation(self):
        """Get environment observation"""
        distance_to_target = np.linalg.norm(self._state - self._target)
        direction_to_target = (self._target - self._state) / max(distance_to_target, 1e-8)
        
        return {
            'agent_position': self._state.copy(),
            'target_position': self._target.copy(),
            'distance_to_target': np.array([distance_to_target], dtype=np.float32),
            'direction_to_target': direction_to_target.astype(np.float32),
            'step_count': np.array([self._step_count], dtype=np.int32),
            'episode_progress': np.array([self._step_count / self.max_episode_steps], dtype=np.float32)
        }
    
    def _validate_observation(self, observation):
        """Validate observation against specification"""
        spec = self.observation_spec()
        
        for key, expected_spec in spec.items():
            if key not in observation:
                raise ValueError(f"Missing observation key: {key}")
            
            obs_value = observation[key]
            if obs_value.shape != expected_spec.shape:
                raise ValueError(
                    f"Observation {key} shape mismatch: {obs_value.shape} != {expected_spec.shape}"
                )
            
            if obs_value.dtype != expected_spec.dtype:
                raise ValueError(
                    f"Observation {key} dtype mismatch: {obs_value.dtype} != {expected_spec.dtype}"
                )
    
    def _validate_action(self, action):
        """Validate action against specification"""
        action_spec = self.action_spec()
        
        if not isinstance(action, (int, np.integer)):
            raise ValueError(f"Action must be integer, got {type(action)}")
        
        if not (0 <= action < action_spec.num_values):
            raise ValueError(
                f"Action {action} out of range [0, {action_spec.num_values})"
            )
    
    def get_metrics(self):
        """Get environment metrics"""
        if not self._episode_rewards:
            return {}
        
        episode_rewards = np.array(self._episode_rewards)
        episode_lengths = np.array(self._episode_lengths) if self._episode_lengths else np.array([0])
        
        return {
            'episodes_completed': self._episode_count,
            'total_steps': len(self._episode_rewards),
            'mean_episode_reward': np.mean(episode_rewards),
            'std_episode_reward': np.std(episode_rewards),
            'mean_episode_length': np.mean(episode_lengths),
            'success_rate': np.mean([r > 5.0 for r in episode_rewards]),
            'action_distribution': {k: v / max(sum(self._action_counts.values()), 1) 
                                   for k, v in self._action_counts.items()}
        }
    
    def observation_spec(self):
        return {
            'agent_position': dm_env.specs.BoundedArray(
                shape=(2,), dtype=np.float32, minimum=-2.0, maximum=2.0, name='position'
            ),
            'target_position': dm_env.specs.BoundedArray(
                shape=(2,), dtype=np.float32, minimum=-2.0, maximum=2.0, name='target'
            ),
            'distance_to_target': dm_env.specs.BoundedArray(
                shape=(1,), dtype=np.float32, minimum=0.0, maximum=4.0, name='distance'
            ),
            'direction_to_target': dm_env.specs.BoundedArray(
                shape=(2,), dtype=np.float32, minimum=-1.0, maximum=1.0, name='direction'
            ),
            'step_count': dm_env.specs.BoundedArray(
                shape=(1,), dtype=np.int32, minimum=0, maximum=self.max_episode_steps, name='steps'
            ),
            'episode_progress': dm_env.specs.BoundedArray(
                shape=(1,), dtype=np.float32, minimum=0.0, maximum=1.0, name='progress'
            )
        }
    
    def action_spec(self):
        return dm_env.specs.DiscreteArray(num_values=4, name='movement_direction')

# Simple agent for demonstration
class SimpleAgent:
    """Simple agent with basic policy and logging"""
    
    def __init__(self, name="SimpleAgent"):
        self.name = name
        self.step_count = 0
        self.episode_count = 0
        self.policy_type = "directional"  # Move towards target
        
        # Performance tracking
        self.episode_rewards = []
        self.episode_lengths = []
        
    def select_action(self, timestep):
        """Select action based on simple policy"""
        self.step_count += 1
        
        if timestep.first():
            self.episode_count += 1
            self.current_episode_reward = 0.0
            self.current_episode_length = 0
        
        # Update tracking
        if timestep.reward is not None:
            self.current_episode_reward += timestep.reward
        
        self.current_episode_length += 1
        
        if timestep.last():
            self.episode_rewards.append(self.current_episode_reward)
            self.episode_lengths.append(self.current_episode_length)
        
        # Simple policy: move towards target
        obs = timestep.observation
        direction = obs['direction_to_target']
        
        # Choose action based on strongest direction component
        if abs(direction[1]) > abs(direction[0]):
            # Move vertically
            action = 0 if direction[1] > 0 else 2  # Up or Down
        else:
            # Move horizontally  
            action = 1 if direction[0] > 0 else 3  # Right or Left
        
        return action
    
    def get_performance_metrics(self):
        """Get agent performance metrics"""
        if not self.episode_rewards:
            return {}
        
        rewards = np.array(self.episode_rewards)
        lengths = np.array(self.episode_lengths)
        
        return {
            'episodes_completed': len(self.episode_rewards),
            'mean_reward': np.mean(rewards),
            'std_reward': np.std(rewards),
            'best_reward': np.max(rewards),
            'worst_reward': np.min(rewards),
            'mean_length': np.mean(lengths),
            'total_steps': self.step_count
        }

print("=== Production Agent-Environment Loop ===")

# Setup environment and agent
env_config = {
    'name': 'TargetReachingEnv',
    'max_episode_steps': 50,
    'target_reward': 100.0
}

environment = ProductionEnvironment(env_config)
agent = SimpleAgent("NavigationAgent")

# Run training loop
num_episodes = 5
print(f"Running {num_episodes} episodes...")

for episode in range(num_episodes):
    print(f"\n--- Episode {episode + 1} ---")
    
    timestep = environment.reset()
    episode_steps = 0
    episode_reward = 0.0
    
    while not timestep.last():
        # Agent selects action
        action = agent.select_action(timestep)
        
        # Environment steps
        timestep = environment.step(action)
        episode_steps += 1
        
        if timestep.reward is not None:
            episode_reward += timestep.reward
        
        # Log every few steps
        if episode_steps % 10 == 0:
            obs = timestep.observation
            distance = obs['distance_to_target'][0]
            progress = obs['episode_progress'][0]
            print(f"  Step {episode_steps}: distance={distance:.3f}, progress={progress:.2f}")
    
    # Final agent call for episode end
    agent.select_action(timestep)
    
    print(f"  Episode result: {episode_steps} steps, reward={episode_reward:.3f}")

# Display final metrics
print(f"\n=== Final Performance Metrics ===")

# Environment metrics
env_metrics = environment.get_metrics()
print(f"Environment Metrics:")
for key, value in env_metrics.items():
    if isinstance(value, dict):
        print(f"  {key}:")
        for sub_key, sub_value in value.items():
            print(f"    {sub_key}: {sub_value:.3f}")
    else:
        print(f"  {key}: {value:.3f}" if isinstance(value, float) else f"  {key}: {value}")

# Agent metrics
agent_metrics = agent.get_performance_metrics()
print(f"\nAgent Metrics:")
for key, value in agent_metrics.items():
    print(f"  {key}: {value:.3f}" if isinstance(value, float) else f"  {key}: {value}")

# Performance analysis
success_rate = env_metrics.get('success_rate', 0.0)
mean_reward = agent_metrics.get('mean_reward', 0.0)

print(f"\n=== Performance Analysis ===")
print(f"Success Rate: {success_rate:.1%}")
print(f"Average Reward: {mean_reward:.3f}")
print(f"Training Efficiency: {success_rate * mean_reward:.3f}")

if success_rate > 0.6:
    print("✅ Good performance achieved!")
elif success_rate > 0.3:
    print("⚠️  Moderate performance, could be improved")
else:
    print("❌ Poor performance, needs significant improvement")

print("\n✅ Production agent-environment loop demonstrated!")
print("Complete RL training pipeline with monitoring, validation, and metrics tracking")